# Data Cleaning - Locaux Location Marrakech
This notebook focuses on cleaning the locaux location data for Marrakech.

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# File path
file_path = '../../data/marrakech_immo_location/locaux_location.csv'

# Load data
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    display(df.head())
else:
    print(f"ERROR: File not found at {file_path}")
    print(f"Current working directory: {os.getcwd()}")

Successfully loaded ../../data/marrakech_immo_location/locaux_location.csv


,id,titre,prix,localisation,type_bien,surface,chambres,salles_bain,description,agence,...,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,quartier,prix_m2,prix_m2_median_quartier
0,no-id-0,Propriété à M'Hamidi,7 000 DH,"M'Hamidi, Marrakech",Appartement,27 m²,NaN,NaN,NaN,Yakeey,...,-1,NaN,7000.0,27.0,0.0,0.0,1.0,M'hamid,259.259259,166.666667
1,no-id-1,Propriété à M'Hamid,7 000 DH,"M'Hamid, Marrakech",Appartement,47 m²,NaN,NaN,NaN,Yakeey,...,-1,NaN,7000.0,47.0,0.0,0.0,1.0,M'hamid,148.936170,166.666667
2,no-id-2,Propriété à Jnan Aourad,25 000 DH,"Jnan Aourad, Marrakech",Appartement,120 m²,NaN,NaN,NaN,Yakeey,...,-1,NaN,25000.0,120.0,0.0,0.0,1.0,Autre,208.333333,180.625000
3,no-id-3,Propriété à Jardin De La Koutoubia,30 000 DH,"Jardin De La Koutoubia, Marrakech",Appartement,130 m²,NaN,NaN,NaN,Yakeey,...,-1,NaN,30000.0,130.0,0.0,0.0,1.0,Autre,230.769231,180.625000
4,no-id-4,Propriété à Quartier Industriel De Sidi Ghanem,38 000 DH,"Quartier Industriel De Sidi Ghanem, Marrakech",Appartement,350 m²,NaN,NaN,NaN,Yakeey,...,-1,NaN,38000.0,350.0,0.0,0.0,1.0,Autre,108.571429,180.625000


## 1. Initial Data Overview

In [30]:
if 'df' in locals():
    print("Shape:", df.shape)
    df.info()
else:
    print("DataFrame 'df' not loaded. Please fix the file path above.")

Shape: (935, 34)
<class 'pandas.DataFrame'>
RangeIndex: 935 entries, 0 to 934
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       350 non-null    str    
 1   titre                    935 non-null    str    
 2   prix                     935 non-null    str    
 3   localisation             935 non-null    str    
 4   type_bien                935 non-null    str    
 5   surface                  859 non-null    str    
 6   chambres                 84 non-null     str    
 7   salles_bain              626 non-null    str    
 8   description              911 non-null    str    
 9   agence                   346 non-null    str    
 10  url                      935 non-null    str    
 11  source                   935 non-null    str    
 12  piscine                  935 non-null    int64  
 13  parking                  935 non-null    int64  
 14  ascenseur           

In [31]:
if 'df' in locals():
    cols_to_drop = ['id', 'url', 'source', 'titre', 'description', 'prix', 'surface', 'localisation', 'chambres', 'salles_bain', 'surface_terrain']
    existing_drops = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=existing_drops)
    print(f"Dropped columns: {existing_drops}")
    print(f"Remaining columns: {df.columns.tolist()}")

Dropped columns: ['id', 'url', 'source', 'titre', 'description', 'prix', 'surface', 'localisation', 'chambres', 'salles_bain', 'surface_terrain']
Remaining columns: ['type_bien', 'agence', 'piscine', 'parking', 'ascenseur', 'terrasse', 'jardin', 'climatisation', 'securite', 'vue', 'meuble', 'neuf', 'cave', 'hammam', 'etage', 'prix_num', 'surface_num', 'chambres_num', 'salles_bain_num', 'nb_pieces', 'quartier', 'prix_m2', 'prix_m2_median_quartier']


## 2. Handling Missing Values

In [32]:
if 'df' in locals():
    # Checking for null values
    null_counts = df.isnull().sum()
    print("Columns with null values before cleaning:")
    print(null_counts[null_counts > 0])
    
    # 1. Variables numériques : imputation par la médiane
    num_cols_to_fill = ['prix_num', 'surface_num', 'prix_m2', 'prix_m2_median_quartier']
    for col in num_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
            
    # 2. Variables catégorielles : imputation par le mode (valeur la plus fréquente)
    cat_cols_mode = ['agence', 'type_bien', 'localisation']
    for col in cat_cols_mode:
        if col in df.columns:
            mode_val = df[col].mode()
            if not mode_val.empty:
                df[col] = df[col].fillna(mode_val[0])
            
    # 3. Champs texte : on met 'Non spécifié'
    text_cols = ['titre', 'description', 'prix', 'surface', 'url']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('Non spécifié')
    
    # 4. Traitement des chambres et salles de bain       # 2. Rooms/Baths logic - Imputation par la moyenne au lieu de 0.0
for col in ['chambres_num', 'salles_bain_num']:
    if col in df.columns:
        # On remplace les 0 par NaN pour qu'ils soient imputés par la moyenne
        df[col] = df[col].replace(0, np.nan)
        # Remplissage par la moyenne (arrondie à l'entier le plus proche)
        df[col] = df[col].fillna(round(df[col].mean() if not df[col].isnull().all() else 1))


# 1b. Calcul du prix_m2 manquant à partir du prix_num et surface_num
if 'prix_m2' in df.columns and 'prix_num' in df.columns and 'surface_num' in df.columns:
    # On remplace temporairement les surfaces 0 par NaN pour éviter la division par zéro
    temp_surface = df['surface_num'].replace(0, np.nan)
    df['prix_m2'] = df['prix_m2'].fillna(df['prix_num'] / temp_surface)
  
    
    # id: on supprime les lignes sans ID car c'est crucial
    if 'id' in df.columns:
        df.dropna(subset=['id'], inplace=True)
        
    print("\nColumns with null values after cleaning:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

Columns with null values before cleaning:
agence                     589
prix_num                   123
surface_num                 97
prix_m2                    196
prix_m2_median_quartier      3
dtype: int64

Columns with null values after cleaning:
Series([], dtype: int64)


In [36]:
df.shape

(842, 23)

## 3. Removing Duplicates

In [33]:
if 'df' in locals():
    initial_size = len(df)
    df = df.drop_duplicates()
    print(f"Removed {initial_size - len(df)} duplicate rows.")

Removed 59 duplicate rows.


## 4. Cleaning Numerical Columns
Handling outliers for price and surface.

In [34]:
if 'df' in locals():
    if 'prix_num' in df.columns:
        q_low = df['prix_num'].quantile(0.01)
        q_hi  = df['prix_num'].quantile(0.99)
        df = df[(df['prix_num'] <= q_hi) & (df['prix_num'] >= q_low)]
        print(f"Data points after price outlier removal: {len(df)}")
    
    if 'surface_num' in df.columns:
        q_low_surf = df['surface_num'].quantile(0.01)
        q_hi_surf  = df['surface_num'].quantile(0.99)
        df = df[(df['surface_num'] <= q_hi_surf) & (df['surface_num'] >= q_low_surf)]
        print(f"Data points after surface outlier removal: {len(df)}")

Data points after price outlier removal: 859
Data points after surface outlier removal: 842


## 5. Saving Cleaned Data

In [35]:
if 'df' in locals():
    output_dir = '../../data/cleaned_data/location'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_file = 'locaux_location_cleaned.csv'
    output_path = os.path.join(output_dir, output_file)
    df.to_csv(output_path, index=False)
    print(f"Cleaned data saved to {output_path}")

Cleaned data saved to ../../data/cleaned_data/location/locaux_location_cleaned.csv
